In [11]:
from bs4 import BeautifulSoup # extract data from html easily
import requests #to send &fetch  http requests to website and retrieve some of their data


In [3]:
# Obtaining properties links
def get_products_links(location, page):
  if page == 1:
    bayut_url =f"https://www.bayut.eg/en/{location}/apartments-for-sale/"
  else:
    bayut_url =f"https://www.bayut.eg/en/{location}/apartments-for-sale/page-{page}/"
  
  response = requests.get(bayut_url) #request the website
  soup = BeautifulSoup(response.text, 'html') #parse the html

  property_links=[]
  links = soup.find_all('a', href=True)
  for link in links:
    link_href =link['href']
    
    if link_href.startswith('/en/property/'):
      if "https" in link_href:
        print(link_href)
      else:
        print("https://www.bayut.eg"+link_href)
    property_links.append(link_href)

  return property_links
   
for page_number in range(1,10):
  get_products_links("cairo",page_number)


https://www.bayut.eg/en/property/details-501997530.html
https://www.bayut.eg/en/property/details-501997530.html
https://www.bayut.eg/en/property/details-501997095.html
https://www.bayut.eg/en/property/details-501997095.html
https://www.bayut.eg/en/property/details-501996696.html
https://www.bayut.eg/en/property/details-501996696.html
https://www.bayut.eg/en/property/details-501996680.html
https://www.bayut.eg/en/property/details-501996680.html
https://www.bayut.eg/en/property/details-501996628.html
https://www.bayut.eg/en/property/details-501996628.html
https://www.bayut.eg/en/property/details-501997197.html
https://www.bayut.eg/en/property/details-501997197.html
https://www.bayut.eg/en/property/details-501997080.html
https://www.bayut.eg/en/property/details-501997080.html
https://www.bayut.eg/en/property/details-501995984.html
https://www.bayut.eg/en/property/details-501995984.html
https://www.bayut.eg/en/property/details-501995556.html
https://www.bayut.eg/en/property/details-5019955

In [4]:
import re
import json
import requests

def get_property_details(property_link):
    HEADERS = {
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Accept-language": "en-US,en;q=0.9",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36"
    }
    
    
    response = requests.get(property_link, headers=HEADERS, timeout =10)
    soup = BeautifulSoup(response.text, 'html.parser')
    scripts = soup.find_all('script')
    json_data = None  

    for script in scripts:
        if script.string and "window.state =" in script.string:
            match_json = re.search(r"window\.state\s*=\s*(\{.*?\})\s*;", script.string, re.DOTALL)
            if match_json:
                json_text = match_json.group(1)
                try:
                    json_data = json.loads(json_text)
                    #print(json.dumps(json_data, indent=2))
                    break
                except json.JSONDecodeError as e:
                    print(f"JSON decoding error: {e}")
                    print(f"Problematic JSON snippet:\n{json_text[:300]}...")  # Print a preview
                    break
    property_data = {
        "Ref. No":soup.find("span", {"aria-label": "Reference"}).get_text(strip=True),
        "title": (soup.find("h1").get_text(strip=True) if soup.find("h1") else "N/A"),
        "type" :soup.find("span", {"aria-label": "Type"}).get_text(strip=True),
        "price" : soup.find("span", {"aria-label": "Price"}).get_text(strip=True),
        "bedrooms": (soup.find("span", {"aria-label": "Beds"}).get_text(strip=True) 
                   if soup.find("span", {"aria-label": "Beds"}) else "N/A"),
        "bathrooms": (soup.find("span", {"aria-label": "Baths"}).get_text(strip=True) 
                    if soup.find("span", {"aria-label": "Baths"}) else "N/A"),
        "area": (soup.find("span", {"aria-label": "Area"}).get_text(strip=True) 
                   if soup.find("span", {"aria-label": "Area"}) else "N/A"),
     
        
    }

    print(json.dumps(property_data, indent=2, ensure_ascii=False))
  
   
# Call the function with the property URL
get_property_details("https://www.bayut.eg/en/property/details-501957375.html")

{
  "Ref. No": "Bayut - Mi060",
  "title": "Prime location in the heart of Smouha in front of Green Plaza, overlooking the club and the plaza, 131-square-meter apartment, ultra-luxurious finishi",
  "type": "Apartment",
  "price": "6,353,500",
  "bedrooms": "3 Beds",
  "bathrooms": "2 Baths",
  "area": "131 Sq. M."
}


In [9]:
# Example usage
if __name__ == "__main__":
    base_url = "https://www.bayut.eg/en/property/details-501957375.html"
    details = get_property_details(base_url)
    if details:
        print(json.dumps(details, indent=2))

{
  "Ref. No": "Bayut - Mi060",
  "title": "Prime location in the heart of Smouha in front of Green Plaza, overlooking the club and the plaza, 131-square-meter apartment, ultra-luxurious finishi",
  "type": "Apartment",
  "price": "6,353,500",
  "bedrooms": "3 Beds",
  "bathrooms": "2 Baths",
  "area": "131 Sq. M."
}


## Debugging

In [ ]:
HEADERS = {
        "Accept": "*/*",
        "Accept-Encoding": "gzip, deflate, br, zstd",
        "Accept-language": "en-US,en;q=0.9",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36"
    }
    
response = requests.get("https://www.bayut.eg/en/property/details-501957375.html", headers=HEADERS, timeout=10)
print(f"Status Code: {response.status_code}")  # Should be 200
print(f"Content Length: {len(response.text)}")  # Should be > 0

In [ ]:

soup = BeautifulSoup(response.text, 'html.parser')
scripts = soup.find_all('script')
print(f"Found {len(scripts)} script tags")
for i, script in enumerate(scripts[:5]):  # Just check first 5
    print(f"Script {i}: {str(script.string)[:100]}...")